# NIST TN 1822 — Verif.2.1: Speed in a corridor

Single agent walks 40 m at 1.0 m/s. The corridor is extended to 60 m with 10 m acceleration and 10 m isolation buffers; see MODIFICATIONS.md.

In [1]:
from datetime import datetime

print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

Executed on 23 May 2026, 09:37 UTC


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from jupedsim_scenarios import load_scenario, run_scenario
from shapely.geometry import Point, Polygon

In [3]:
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#f7f7f5",
        "axes.edgecolor": "#3a3a3a",
        "axes.labelcolor": "#1d1d1d",
        "axes.titleweight": "bold",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "font.size": 11,
        "figure.figsize": (8, 5),
    }
)

## Load and run the scenario

In [4]:
SCENARIO_ZIP = Path("scenario_files") / "Nist-2-1-corridor-speed.zip"
scenario = load_scenario(str(SCENARIO_ZIP))
print(scenario.summary())
result = run_scenario(scenario, seed=42)

Scenario: /work/standards/nist/scenario_files/Nist-2-1-corridor-speed.zip
  Model:         CollisionFreeSpeedModel
  Seed:          42
  Max time:      100s
  Exits:         1
  Distributions: 1
  Stages:        0
  Zones:         0
  Journeys:      1
  Agents:        ~1
  Journey elems: 2
  Route:         1 distribution, 0 checkpoint, 1 exit
  Sequence:      jps-distributions_0 -> jps-exits_0
    jps-distributions_0: 1 agents
Using fallback logic: No journeys defined
Processing with parameters: {'number': 1, 'radius': 0.15, 'v0': 1.0, 'distribution_mode': 'by_number', 'radius_distribution': 'constant', 'v0_distribution': 'constant', 'use_flow_spawning': False}
Using default parameters: v0=1.0, radius=0.15, n_agents=1

Distribution jps-distributions_0: {'number': 1, 'radius': 0.15, 'v0': 1.0, 'distribution_mode': 'by_number', 'percentage': None, 'use_flow_spawning': False, 'flow_start_time': 0, 'flow_end_time': 10, 'use_premovement': False, 'premovement_distribution': 'gamma', 'premove

## Measure walking speed across the 40 m segment

In [5]:
df = result.trajectory_dataframe().sort_values(["id", "frame"])
MEAS_START_X = 10.0
MEAS_END_X = 50.0
DIRECTION = "right"
rows = []
for agent_id, sub in df.groupby("id"):
    sub = sub.reset_index(drop=True)
    if DIRECTION == "right":
        t_in_idx = sub[sub.x >= MEAS_START_X].index.min()
        t_out_idx = sub[sub.x >= MEAS_END_X].index.min()
    else:
        t_in_idx = sub[sub.x <= MEAS_START_X].index.min()
        t_out_idx = sub[sub.x <= MEAS_END_X].index.min()
    if pd.isna(t_in_idx) or pd.isna(t_out_idx):
        continue
    t_in = sub.loc[t_in_idx, "frame"] / result.frame_rate
    t_out = sub.loc[t_out_idx, "frame"] / result.frame_rate
    rows.append({"id": int(agent_id), "t_in_s": t_in, "t_out_s": t_out, "transit_s": t_out - t_in})
transit = pd.DataFrame(rows)
transit["speed_m_s"] = abs(MEAS_END_X - MEAS_START_X) / transit["transit_s"]
transit

,id,t_in_s,t_out_s,transit_s,speed_m_s
0,1,8.8,48.8,40.0,1.0


## Plot trajectory x vs t

In [6]:
fig, ax = plt.subplots()
for agent_id, sub in df.groupby("id"):
    ax.plot(sub.frame / result.frame_rate, sub.x, label=f"agent {agent_id}")
ax.axhline(MEAS_START_X, color="k", ls="--", alpha=0.3, label="measure start")
ax.axhline(MEAS_END_X, color="k", ls=":", alpha=0.3, label="measure end")
ax.set_xlabel("time [s]")
ax.set_ylabel("x position [m]")
ax.legend()
plt.show()

## Acceptance

In [7]:
TARGET = 1.0
TOL = 0.05
observed = transit["speed_m_s"].mean()
print(f"observed mean speed = {observed:.3f} m/s; target = {TARGET} m/s")
assert abs(observed - TARGET) <= TOL, (observed, TARGET)

observed mean speed = 1.000 m/s; target = 1.0 m/s


In [8]:
result.cleanup()